# Chapter 12 probe (agent run): does vLLM load `oopere/llama-3.2-3b-attn-drop-6`?

Target: Llama-3.2-3B with attention removed from blocks 18, 20, 21, 22, 23 and 24 (22 of 28 blocks keep attention).

**This run is L4 only.** The first cell stops if the GPU is not an L4.

Protocol:
1. Install a pinned vLLM (starting with 0.30.0) and record vLLM, torch, transformers, CUDA and GPU versions.
2. **Control first**: load an unmodified Llama-3.2-3B with the exact same settings (`dtype="half"`, `max_model_len=2048`, `gpu_memory_utilization=0.85`, `enforce_eager=True`, `trust_remote_code=True`).
3. If the control fails, the problem is the environment. Try at most two older vLLM releases, one attempt each, and never test the target on a version where the control failed.
4. If the control loads, load the target with the same settings.
5. If the target fails, extract the first real error from the full log.
6. If the target loads, test structured outputs, batch-1 latency (median of 10 after a warm-up) and KV cache bytes per token against the formula.

Run it from the first cell. The kernel never imports torch or vLLM; everything that touches them runs in a subprocess. Every result and log is written under `/content/ch12_probe/`, so a kernel restart loses nothing: rerunning skips steps that already have a saved result.

In [ ]:
import json, os, re, subprocess, sys, time
from pathlib import Path

WORK_DIR = Path("/content/ch12_probe")
LOG_DIR = WORK_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = WORK_DIR / "results.json"

# Stop right away if this is not the L4: this phase must not run on a T4
gpu_line = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                           "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu_line)
assert "L4" in gpu_line, f"This phase runs only on an L4, found: {gpu_line!r}"

In [ ]:
TARGET_ID = "oopere/llama-3.2-3b-attn-drop-6"
CONTROL_CANDIDATES = ["meta-llama/Llama-3.2-3B", "unsloth/Llama-3.2-3B"]

PRIMARY_VLLM = "0.30.0"
MAX_VLLM_VERSIONS = 3          # primary + at most two older fallbacks
LOAD_TIMEOUT_S = 1800

# One shared configuration for control and target, so the comparison is fair.
# float16 (not bfloat16) keeps results comparable with a future T4 run.
LLM_KWARGS = dict(dtype="half", max_model_len=2048, gpu_memory_utilization=0.85,
                  enforce_eager=True, trust_remote_code=True)

# KV cache formula inputs (Llama-3.2-3B)
KV_HEADS, HEAD_DIM, BYTES_FP16 = 8, 128, 2
TOTAL_BLOCKS, ATTN_BLOCKS = 28, 22
PER_LAYER_BYTES = 2 * KV_HEADS * HEAD_DIM * BYTES_FP16      # K + V
FORMULA = {"attention_blocks_only": PER_LAYER_BYTES * ATTN_BLOCKS,   # 90,112
           "all_blocks": PER_LAYER_BYTES * TOTAL_BLOCKS}             # 114,688

# Optional HF token from Colab secrets (needed only if the Meta repo is used)
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN") or "")
except Exception:
    pass

def load_results():
    return json.loads(RESULTS_PATH.read_text()) if RESULTS_PATH.exists() else {"versions": {}}

def save_results(res):
    RESULTS_PATH.write_text(json.dumps(res, indent=2))

results = load_results()
results["formula_bytes_per_token"] = FORMULA
results["llm_kwargs"] = LLM_KWARGS
save_results(results)
print(json.dumps(FORMULA, indent=2))

## 1. Pick the control model (no gated repo needed)

In [ ]:
from huggingface_hub import hf_hub_download

def first_accessible(repo_ids):
    # Downloading config.json confirms the repo exists and that we can read it
    for repo in repo_ids:
        try:
            hf_hub_download(repo, "config.json", token=os.environ.get("HF_TOKEN") or None)
            return repo
        except Exception as e:
            print(f"{repo}: not usable ({type(e).__name__}: {str(e)[:150]})")
    raise RuntimeError("No accessible Llama-3.2-3B control checkpoint")

if "control_model" not in results:
    results["control_model"] = first_accessible(CONTROL_CANDIDATES)
    save_results(results)

CONTROL_ID = results["control_model"]
cfg = json.load(open(hf_hub_download(CONTROL_ID, "config.json")))
print("Control model:", CONTROL_ID)
print({k: cfg.get(k) for k in ["num_hidden_layers", "num_key_value_heads", "head_dim", "hidden_size"]})

## 2. Probe script (runs in its own process for every load)

In [ ]:
%%writefile /content/ch12_probe/vllm_probe.py
import json, sys, time, statistics, traceback

model_id, out_path, mode, kwargs_json = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4]
LLM_KWARGS = json.loads(kwargs_json)
result = {"model": model_id, "mode": mode, "loaded": False}

SCHEMA = {
    "type": "object",
    "properties": {
        "name": {"const": "search"},
        "arguments": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
            "additionalProperties": False,
        },
    },
    "required": ["name", "arguments"],
    "additionalProperties": False,
}
PROMPT = (
    "Convert the request into a JSON tool call.\n"
    "Request: look up the weather in Paris\n"
    'Call: {"name": "search", "arguments": {"query": "weather in Paris"}}\n'
    "Request: find the capital of Peru\n"
    "Call:"
)
N_TOKENS = 30

def dump():
    json.dump(result, open(out_path, "w"), indent=2)

try:
    import vllm
    from vllm import LLM, SamplingParams
    result["vllm_version"] = vllm.__version__

    t0 = time.perf_counter()
    llm = LLM(model=model_id, **LLM_KWARGS)
    result["load_seconds"] = round(time.perf_counter() - t0, 1)
    result["loaded"] = True
    print("PROBE: model loaded", flush=True)
    dump()

    free = SamplingParams(temperature=0.0, max_tokens=N_TOKENS, ignore_eos=True)
    result["free_text"] = llm.generate([PROMPT], free)[0].outputs[0].text   # also the warm-up

    if mode == "target":
        times = []
        for _ in range(10):
            t0 = time.perf_counter()
            llm.generate([PROMPT], free, use_tqdm=False)
            times.append((time.perf_counter() - t0) * 1000)
        result["latency_batch1"] = {
            "n_new_tokens": N_TOKENS,
            "p50_ms_request": round(statistics.median(times), 1),
            "p50_ms_per_token": round(statistics.median(times) / N_TOKENS, 2),
            "max_ms_per_token": round(max(times) / N_TOKENS, 2),
            "runs_ms": [round(t, 1) for t in times],
        }

        # The structured outputs API changed names across vLLM releases
        try:
            from vllm.sampling_params import StructuredOutputsParams
            guided = SamplingParams(temperature=0.0, max_tokens=64,
                                    structured_outputs=StructuredOutputsParams(json=SCHEMA))
            result["structured_api"] = "SamplingParams(structured_outputs=StructuredOutputsParams(json=...))"
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            guided = SamplingParams(temperature=0.0, max_tokens=64,
                                    guided_decoding=GuidedDecodingParams(json=SCHEMA))
            result["structured_api"] = "SamplingParams(guided_decoding=GuidedDecodingParams(json=...))"
        text = llm.generate([PROMPT], guided)[0].outputs[0].text
        result["structured_text"] = text
        try:
            parsed = json.loads(text)
            result["structured_valid_json"] = True
            result["structured_matches_schema"] = (
                parsed.get("name") == "search"
                and isinstance(parsed.get("arguments", {}).get("query"), str)
                and set(parsed) == {"name", "arguments"}
                and set(parsed["arguments"]) == {"query"})
        except Exception:
            result["structured_valid_json"] = False
            result["structured_matches_schema"] = False
except Exception:
    result["error_tail"] = traceback.format_exc()[-4000:]
    traceback.print_exc()

dump()

## 3. Helpers: install, record versions, run a load, read the log

In [ ]:
def pip_install_vllm(version):
    log = LOG_DIR / f"pip_vllm_{version}.log"
    with open(log, "w") as f:
        proc = subprocess.run([sys.executable, "-m", "pip", "install", f"vllm=={version}", "huggingface_hub"],
                              stdout=f, stderr=subprocess.STDOUT, text=True)
    return proc.returncode, str(log)

ENV_SNIPPET = r'''
import json, torch, transformers, vllm
print(json.dumps({
    "vllm": vllm.__version__, "torch": torch.__version__,
    "transformers": transformers.__version__, "cuda_torch": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0), "capability": list(torch.cuda.get_device_capability(0)),
}))
'''

def record_env():
    # Read versions in a fresh process: the kernel's own imports may be stale after pip
    proc = subprocess.run([sys.executable, "-c", ENV_SNIPPET], capture_output=True, text=True)
    if proc.returncode != 0:
        return {"error": proc.stderr[-2000:]}
    env = json.loads(proc.stdout.strip().splitlines()[-1])
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    m = re.search(r"CUDA Version:\s*([\d.]+)", smi)
    env["cuda_driver"] = m.group(1) if m else None
    return env

def torch_version_now():
    p = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                       capture_output=True, text=True)
    return p.stdout.strip() or None

def run_probe(model_id, mode, version, tag_suffix="", extra_kwargs=None, script="vllm_probe.py"):
    tag = f"vllm{version}_{mode}{tag_suffix}"
    kwargs = {**LLM_KWARGS, **(extra_kwargs or {})}
    out_json, log_path = WORK_DIR / f"{tag}.json", LOG_DIR / f"{tag}.log"
    if out_json.exists():
        out_json.unlink()
    t0 = time.time()
    with open(log_path, "w") as f:
        try:
            proc = subprocess.run(
                [sys.executable, str(WORK_DIR / script), model_id, str(out_json), mode,
                 json.dumps(kwargs)],
                stdout=f, stderr=subprocess.STDOUT, text=True, timeout=LOAD_TIMEOUT_S)
            rc = proc.returncode
        except subprocess.TimeoutExpired:
            rc = "timeout"
    res = json.loads(out_json.read_text()) if out_json.exists() else {"loaded": False}
    res.update({"returncode": rc, "wall_seconds": round(time.time() - t0, 1), "log": str(log_path),
                "llm_kwargs": kwargs})
    res["kv_cache"] = parse_kv(log_path.read_text(errors="replace"))
    if not res["loaded"]:
        res["root_cause"] = first_real_error(log_path)
    return res

def parse_kv(log_text):
    mem = re.search(r"Available KV cache memory:\s*([\d.]+)\s*GiB", log_text)
    tok = re.search(r"GPU KV cache size:\s*([\d,]+)\s*tokens", log_text)
    if not (mem and tok):
        return None
    gib, tokens = float(mem.group(1)), int(tok.group(1).replace(",", ""))
    return {"available_gib": gib, "tokens": tokens,
            "bytes_per_token": round(gib * 2**30 / tokens),
            "lines": [mem.group(0), tok.group(0)]}

ERROR_PATTERNS = re.compile(r"Error|Exception|KeyError|not supported|unsupported|shape mismatch|"
                            r"size mismatch|attention|Traceback", re.IGNORECASE)
WRAPPER_NOISE = re.compile(r"Engine core initialization failed|Failed core proc", re.IGNORECASE)
BENIGN = re.compile(r"warn|deprecat|FutureWarning|UserWarning|INFO .*attention backend|Using .*attention",
                    re.IGNORECASE)

def first_real_error(log_path, context=6):
    # The last line ("Engine core initialization failed") is the wrapper, not the cause.
    # Return the first error-looking line that is neither the wrapper nor a warning,
    # with surrounding lines, plus the first Python exception line for cross-checking.
    lines = Path(log_path).read_text(errors="replace").splitlines()
    first_idx = next((i for i, l in enumerate(lines)
                      if ERROR_PATTERNS.search(l) and not WRAPPER_NOISE.search(l) and not BENIGN.search(l)),
                     None)
    exc_re = re.compile(r"^\s*(?:\(\w+.*?\)\s*)?([A-Za-z_.]*(?:Error|Exception)):\s")
    exc_lines = [(i + 1, l) for i, l in enumerate(lines) if exc_re.match(l) and not WRAPPER_NOISE.search(l)]
    out = {"first_exception_line": exc_lines[0] if exc_lines else None,
           "all_exception_lines": exc_lines[:10]}
    if first_idx is not None:
        lo, hi = max(0, first_idx - context), min(len(lines), first_idx + context + 1)
        out["first_match_line_no"] = first_idx + 1
        out["first_match_context"] = [f"{j + 1}: {lines[j]}" for j in range(lo, hi)]
    return out

## 4. Fallback versions (only used if the control fails on 0.30.0)

In [ ]:
def older_vllm_releases(before, n=2):
    # Latest patch of each of the n minor series just below `before`
    out = subprocess.run([sys.executable, "-m", "pip", "index", "versions", "vllm"],
                         capture_output=True, text=True).stdout
    m = re.search(r"Available versions:\s*(.+)", out)
    if not m:
        return []
    key = lambda v: tuple(int(x) for x in re.findall(r"\d+", v)[:3])
    stable = [v.strip() for v in m.group(1).split(",") if re.fullmatch(r"\d+\.\d+\.\d+", v.strip())]
    by_minor = {}
    for v in sorted(stable, key=key):
        if key(v) < key(before):
            by_minor[key(v)[:2]] = v
    return [by_minor[k] for k in sorted(by_minor, reverse=True)[:n]]

if "fallback_versions" not in results:
    results["fallback_versions"] = older_vllm_releases(PRIMARY_VLLM)
    save_results(results)
VERSION_PLAN = ([PRIMARY_VLLM] + results["fallback_versions"])[:MAX_VLLM_VERSIONS]
print("Version plan (fallbacks used only if the control fails):", VERSION_PLAN)

## 5. Run the protocol

One attempt per vLLM version, at most three versions. The target is loaded only on a version where the control has already loaded. Rerunning this cell picks up saved results and does not repeat attempts; to force a version again, delete its entry in `results.json`.

If installing vLLM changes torch, the cell prints a warning. All loads still run in fresh subprocesses, but **restart the kernel and rerun from the top** before trusting anything imported in the kernel.

In [ ]:
results = load_results()
for version in VERSION_PLAN:
    entry = results["versions"].setdefault(version, {})

    if "install" not in entry:
        torch_before = torch_version_now()
        rc, pip_log = pip_install_vllm(version)
        torch_after = torch_version_now()
        entry["install"] = {"returncode": rc, "log": pip_log,
                            "torch_before": torch_before, "torch_after": torch_after,
                            "torch_changed": torch_before != torch_after}
        save_results(results)
        if entry["install"]["torch_changed"]:
            print(f"WARNING: installing vLLM {version} changed torch {torch_before} -> {torch_after}. "
                  "Restart the kernel and rerun from the first cell.")
    if entry["install"]["returncode"] != 0:
        print(f"vLLM {version}: pip install failed, see {entry['install']['log']}")
        continue

    entry.setdefault("env", record_env())
    save_results(results)
    print(f"\n=== vLLM {version} ===\n", json.dumps(entry["env"], indent=2))

    if "control" not in entry:
        entry["control"] = run_probe(CONTROL_ID, "control", version)
        save_results(results)
    print(f"Control ({CONTROL_ID}) loaded:", entry["control"]["loaded"])

    if not entry["control"]["loaded"]:
        # The environment is broken: the target is not tested on this version
        print("Control failed -> environment problem. Root cause:")
        print(json.dumps(entry["control"].get("root_cause"), indent=2)[:3000])
        continue

    if "target" not in entry:
        entry["target"] = run_probe(TARGET_ID, "target", version)
        save_results(results)
    print(f"Target ({TARGET_ID}) loaded:", entry["target"]["loaded"])
    break    # the control loaded on this version, so this version is the answer
else:
    print("\nThe control did not load on any tested vLLM version. Stopping: this is an environment finding.")

## 6. Results

In [ ]:
results = load_results()
for version, e in results["versions"].items():
    print(f"\n######## vLLM {version} ########")
    print("env:", json.dumps(e.get("env"), indent=1))
    for role in ["control", "target"]:
        r = e.get(role)
        if r is None:
            print(f"[{role}] not run"); continue
        print(f"[{role}] loaded={r['loaded']} returncode={r['returncode']} log={r['log']}")
        if not r["loaded"]:
            rc = r.get("root_cause", {})
            print("  first exception:", rc.get("first_exception_line"))
            print("  first matching line, with context:")
            print("\n".join("    " + l for l in rc.get("first_match_context", [])))
        else:
            print("  KV cache:", r.get("kv_cache"))
            if role == "target":
                print("  free text      :", repr(r.get("free_text")))
                print("  structured API :", r.get("structured_api"))
                print("  structured     :", r.get("structured_valid_json"), r.get("structured_matches_schema"),
                      "->", repr(r.get("structured_text")))
                print("  latency        :", {k: v for k, v in r["latency_batch1"].items() if k != "runs_ms"})
                kv = r.get("kv_cache")
                if kv:
                    for name, ref in FORMULA.items():
                        print(f"  bytes/token measured {kv['bytes_per_token']:,} vs formula {name} {ref:,} "
                              f"(ratio {kv['bytes_per_token'] / ref:.3f})")

In [ ]:
# Write a markdown report to the runtime as well, so it survives the session output
def build_report(res):
    out = ["# vLLM probe report (L4)", ""]
    for version, e in res["versions"].items():
        env = e.get("env", {})
        out.append(f"## vLLM {version}")
        out.append(f"- torch {env.get('torch')}, transformers {env.get('transformers')}, "
                   f"CUDA (torch) {env.get('cuda_torch')}, CUDA (driver) {env.get('cuda_driver')}, GPU {env.get('gpu')}")
        out.append(f"- torch changed by install: {e.get('install', {}).get('torch_changed')}")
        for role in ["control", "target"]:
            r = e.get(role)
            if r is None:
                out.append(f"- {role}: not run"); continue
            line = f"- {role} ({r['model']}): {'LOADED' if r['loaded'] else 'FAILED'}"
            if not r["loaded"]:
                line += f" (first exception: `{(r.get('root_cause') or {}).get('first_exception_line')}`)"
            out.append(line)
            if r["loaded"] and r.get("kv_cache"):
                out.append(f"  - KV cache: {r['kv_cache']['available_gib']} GiB, {r['kv_cache']['tokens']:,} tokens, "
                           f"{r['kv_cache']['bytes_per_token']:,} bytes/token")
            if role == "target" and r["loaded"]:
                out.append(f"  - structured outputs: valid JSON={r.get('structured_valid_json')}, "
                           f"matches schema={r.get('structured_matches_schema')}")
                out.append(f"  - latency batch 1: {r['latency_batch1']['p50_ms_per_token']} ms/token (p50 of 10)")
        out.append("")
    out.append(f"Formula bytes/token: 22 attention blocks = {FORMULA['attention_blocks_only']:,}; "
               f"28 blocks = {FORMULA['all_blocks']:,}")
    return "\n".join(out)

report = build_report(results)
(WORK_DIR / "report.md").write_text(report)
print(report)

## 7. Root-cause check (no GPU, no new vLLM attempt)

The runtime files live on the Colab VM, not on the local machine, so this cell prints them. It also checks what the checkpoint really contains for an attention-free block (18) next to an intact block (17), and which architecture vLLM resolved.

In [ ]:
from huggingface_hub import hf_hub_download

print((WORK_DIR / "report.md").read_text())
print("Files in the runtime:", sorted(str(p.relative_to(WORK_DIR)) for p in WORK_DIR.rglob("*") if p.is_file()))

# Keys the checkpoint ships for an intact block vs an attention-free block
index = json.load(open(hf_hub_download(TARGET_ID, "model.safetensors.index.json")))
keys = index["weight_map"]
for layer in (17, 18):
    print(f"\nlayer {layer}:", sorted(k.split(f"layers.{layer}.")[1] for k in keys if f"layers.{layer}." in k))

# How the repo's custom code builds an attention-free block (Transformers only)
print("\n--- modeling_pruned_llama.py ---")
print(open(hf_hub_download(TARGET_ID, "modeling_pruned_llama.py")).read())

# Which model class vLLM actually used, and the full traceback around the first error
target_log = Path(results["versions"][PRIMARY_VLLM]["target"]["log"]).read_text(errors="replace").splitlines()
print("\n--- architecture lines from the vLLM log ---")
print("\n".join(l for l in target_log if re.search(r"architecture|model_impl|Resolved|Transformers backend", l, re.I)))
print("\n--- log lines 130-160 ---")
print("\n".join(f"{i + 1}: {l}" for i, l in enumerate(target_log) if 130 <= i + 1 <= 160))

## 8. Two ways to serve the attention-free model

Same vLLM (0.30.0), same GPU, same settings as the control, one attempt each.

- **Option 1, the vLLM Transformers backend** (`model_impl="transformers"`). vLLM runs the Hugging Face model class, which with `trust_remote_code=True` could be the repo's `PrunedLlamaForCausalLM`.
- **Option 3, a custom vLLM model class**, installed as a vLLM plugin. It reuses vLLM's own Llama and swaps in a decoder layer that, for blocks listed in `dropped_attn_layers`, builds only `post_attention_layernorm` and the MLP. Those blocks register no `Attention` layer, so vLLM should not reserve KV cache for them. This is the option that can show whether the saving materializes: 90,112 bytes/token expected, against 114,688 for 28 blocks.

A Transformers run of the same prompt (greedy, 30 tokens, float16) is the correctness reference. A configuration that loads but produces different text is not a valid result.

In [ ]:
# Correctness reference: the repo's own Transformers class, greedy, float16
REF_SCRIPT = r'''
import json, sys, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id, out_path = sys.argv[1], sys.argv[2]
PROMPT = (
    "Convert the request into a JSON tool call.\n"
    "Request: look up the weather in Paris\n"
    'Call: {"name": "search", "arguments": {"query": "weather in Paris"}}\n'
    "Request: find the capital of Peru\n"
    "Call:"
)
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="cuda",
                                             trust_remote_code=True).eval()
inputs = tok(PROMPT, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=30, min_new_tokens=30, do_sample=False)
text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
json.dump({"model_class": type(model).__name__, "free_text": text}, open(out_path, "w"), indent=2)
'''
(WORK_DIR / "hf_reference.py").write_text(REF_SCRIPT)

results = load_results()
entry = results["versions"][PRIMARY_VLLM]
if "hf_reference" not in entry:
    ref_json, ref_log = WORK_DIR / "hf_reference.json", LOG_DIR / "hf_reference.log"
    with open(ref_log, "w") as f:
        subprocess.run([sys.executable, str(WORK_DIR / "hf_reference.py"), TARGET_ID, str(ref_json)],
                       stdout=f, stderr=subprocess.STDOUT, text=True, timeout=LOAD_TIMEOUT_S)
    entry["hf_reference"] = (json.loads(ref_json.read_text()) if ref_json.exists()
                             else {"error": ref_log.read_text()[-3000:]})
    save_results(results)
print(json.dumps(entry["hf_reference"], indent=2))

### Option 1: vLLM Transformers backend

In [ ]:
results = load_results()
entry = results["versions"][PRIMARY_VLLM]
if "target_transformers_backend" not in entry:
    entry["target_transformers_backend"] = run_probe(
        TARGET_ID, "target", PRIMARY_VLLM, tag_suffix="_transformers_backend",
        extra_kwargs={"model_impl": "transformers"})
    save_results(results)
r = entry["target_transformers_backend"]
print("loaded:", r["loaded"], "| log:", r["log"])
if not r["loaded"]:
    print(json.dumps(r.get("root_cause"), indent=2)[:4000])

### Option 3: custom vLLM model class (plugin)

The plugin is a small package with a `vllm.general_plugins` entry point. vLLM loads it in every process, including the engine core, which runs in its own process. The checkpoint is untouched: `hf_overrides` only points vLLM to the new architecture name and passes the `dropped_attn_layers` list read from the checkpoint's `config.json`.

In [ ]:
PLUGIN_DIR = WORK_DIR / "attn_drop_plugin"
(PLUGIN_DIR / "attn_drop_vllm").mkdir(parents=True, exist_ok=True)

(PLUGIN_DIR / "pyproject.toml").write_text('''
[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "attn-drop-vllm"
version = "0.1.0"

[project.entry-points."vllm.general_plugins"]
attn_drop = "attn_drop_vllm:register"

[tool.setuptools]
packages = ["attn_drop_vllm"]
''')

(PLUGIN_DIR / "attn_drop_vllm" / "__init__.py").write_text('''
def register():
    from vllm import ModelRegistry
    # Lazy "module:Class" string, so CUDA is not touched when the plugin loads
    if "AttnDropLlamaForCausalLM" not in ModelRegistry.get_supported_archs():
        ModelRegistry.register_model("AttnDropLlamaForCausalLM",
                                     "attn_drop_vllm.model:AttnDropLlamaForCausalLM")
''')

(PLUGIN_DIR / "attn_drop_vllm" / "model.py").write_text('''
# Llama for vLLM where some decoder blocks have no attention sub-block.
# Blocks listed in config.dropped_attn_layers keep only post_attention_layernorm
# and the MLP, matching the checkpoint (no input_layernorm, no q/k/v/o_proj).
# They register no Attention layer, so vLLM allocates no KV cache for them.
from torch import nn

from vllm.model_executor.layers.layernorm import RMSNorm
from vllm.model_executor.models.llama import (
    LlamaAttention, LlamaDecoderLayer, LlamaForCausalLM, LlamaMLP)
from vllm.model_executor.models.utils import extract_layer_index


class AttnFreeAwareDecoderLayer(LlamaDecoderLayer):
    def __init__(self, vllm_config, prefix="", config=None, attn_layer_type=LlamaAttention):
        config = config or vllm_config.model_config.hf_config
        dropped = set(getattr(config, "dropped_attn_layers", None) or [])
        if extract_layer_index(prefix) not in dropped:
            super().__init__(vllm_config=vllm_config, prefix=prefix, config=config,
                             attn_layer_type=attn_layer_type)
            self.has_attention = True
            return

        nn.Module.__init__(self)
        self.has_attention = False
        self.hidden_size = config.hidden_size
        self.mlp = LlamaMLP(
            hidden_size=config.hidden_size,
            intermediate_size=config.intermediate_size,
            hidden_act=config.hidden_act,
            quant_config=self.get_quant_config(vllm_config),
            bias=getattr(config, "mlp_bias", False),
            prefix=f"{prefix}.mlp",
        )
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, positions, hidden_states, residual):
        if self.has_attention:
            return super().forward(positions, hidden_states, residual)
        # vLLM carries the residual stream as (pending delta, residual); the fused
        # add + norm folds the delta in, so this computes x + mlp(norm(x)),
        # the same as the repo's Transformers forward_no_attn
        if residual is None:
            residual = hidden_states
            hidden_states = self.post_attention_layernorm(hidden_states)
        else:
            hidden_states, residual = self.post_attention_layernorm(hidden_states, residual)
        hidden_states = self.mlp(hidden_states)
        return hidden_states, residual


class AttnDropLlamaForCausalLM(LlamaForCausalLM):
    def __init__(self, *, vllm_config, prefix=""):
        if not getattr(vllm_config.model_config.hf_config, "dropped_attn_layers", None):
            raise ValueError("AttnDropLlamaForCausalLM needs config.dropped_attn_layers")
        super().__init__(vllm_config=vllm_config, prefix=prefix,
                         layer_type=AttnFreeAwareDecoderLayer)
''')

with open(LOG_DIR / "pip_attn_drop_plugin.log", "w") as f:
    rc = subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", str(PLUGIN_DIR)],
                        stdout=f, stderr=subprocess.STDOUT).returncode
print("plugin install returncode:", rc)

# Check in a fresh process that vLLM discovers the plugin and the architecture
CHECK = '''
from vllm.plugins import load_general_plugins
from vllm import ModelRegistry
load_general_plugins()
print("AttnDropLlamaForCausalLM" in ModelRegistry.get_supported_archs())
'''
chk = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True)
print("architecture registered:", chk.stdout.strip().splitlines()[-1:] or chk.stderr[-1500:])

In [ ]:
target_cfg = json.load(open(hf_hub_download(TARGET_ID, "config.json")))
DROPPED = target_cfg["dropped_attn_layers"]
print("dropped_attn_layers:", DROPPED)

results = load_results()
entry = results["versions"][PRIMARY_VLLM]
if "target_custom_class" not in entry:
    entry["target_custom_class"] = run_probe(
        TARGET_ID, "target", PRIMARY_VLLM, tag_suffix="_custom_class",
        extra_kwargs={"hf_overrides": {"architectures": ["AttnDropLlamaForCausalLM"],
                                       "dropped_attn_layers": DROPPED}})
    save_results(results)
r = entry["target_custom_class"]
print("loaded:", r["loaded"], "| log:", r["log"])
if not r["loaded"]:
    print(json.dumps(r.get("root_cause"), indent=2)[:4000])

### Comparison

In [ ]:
results = load_results()
entry = results["versions"][PRIMARY_VLLM]
ref_text = entry.get("hf_reference", {}).get("free_text")
control_kv = (entry["control"].get("kv_cache") or {}).get("bytes_per_token")

lines = [f"# Serving options for {TARGET_ID} (vLLM {PRIMARY_VLLM}, {entry['env'].get('gpu')}, float16)", "",
         f"- Transformers reference text: {ref_text!r}",
         f"- Control bytes/token: {control_kv:,}" if control_kv else "- Control bytes/token: n/a",
         f"- Formula: 22 attention blocks = {FORMULA['attention_blocks_only']:,}, 28 blocks = {FORMULA['all_blocks']:,}", ""]
for key, label in [("target", "Native Llama (as is)"),
                   ("target_transformers_backend", "Option 1: Transformers backend"),
                   ("target_custom_class", "Option 3: custom vLLM class")]:
    r = entry.get(key)
    if r is None:
        lines.append(f"## {label}: not run"); continue
    lines.append(f"## {label}: {'LOADED' if r['loaded'] else 'FAILED'}")
    if not r["loaded"]:
        lines.append(f"- first exception: `{(r.get('root_cause') or {}).get('first_exception_line')}`")
        continue
    same = r.get("free_text") == ref_text
    lines.append(f"- output equals Transformers reference: {same}")
    if not same:
        lines.append(f"  - vLLM text: {r.get('free_text')!r}")
    lines.append(f"- structured outputs: API `{r.get('structured_api')}`, valid JSON={r.get('structured_valid_json')}, "
                 f"matches schema={r.get('structured_matches_schema')}, text={r.get('structured_text')!r}")
    lat = r.get("latency_batch1", {})
    lines.append(f"- batch-1 latency: p50 {lat.get('p50_ms_per_token')} ms/token "
                 f"({lat.get('p50_ms_request')} ms per {lat.get('n_new_tokens')}-token request, median of 10)")
    kv = r.get("kv_cache")
    if kv:
        lines.append(f"- KV cache: {kv['available_gib']} GiB, {kv['tokens']:,} tokens, {kv['bytes_per_token']:,} bytes/token "
                     f"(ratio vs 22-block formula {kv['bytes_per_token'] / FORMULA['attention_blocks_only']:.3f}, "
                     f"vs 28-block formula {kv['bytes_per_token'] / FORMULA['all_blocks']:.3f})")
        lines.append(f"- KV log lines: {kv['lines']}")
    lines.append("")

options_report = "\n".join(lines)
(WORK_DIR / "report_options.md").write_text(options_report)
print(options_report)

## 9. Same-engine latency baseline and structured outputs with whitespace control

One more load each for the control and for the custom class (same vLLM, same settings). Each run measures:

- **Batch-1 latency:** same prompt and method as before (30 tokens, `ignore_eos`, warm-up then median of 10). Both models are measured in the same engine, so they can be compared.
- **Structured outputs:** two variants with the same JSON schema:
  - `disable_any_whitespace=True`, `max_tokens=64`: the grammar can no longer pad with whitespace.
  - default whitespace with `max_tokens=256`: does the model close the object if given more room?

  Running both on the control shows whether the whitespace problem comes from the pruning or from the prompt and model family.

In [ ]:
%%writefile /content/ch12_probe/vllm_probe_extra.py
import json, sys, time, statistics, traceback

model_id, out_path, mode, kwargs_json = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4]
LLM_KWARGS = json.loads(kwargs_json)
result = {"model": model_id, "mode": mode, "loaded": False}

SCHEMA = {
    "type": "object",
    "properties": {
        "name": {"const": "search"},
        "arguments": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
            "additionalProperties": False,
        },
    },
    "required": ["name", "arguments"],
    "additionalProperties": False,
}
PROMPT = (
    "Convert the request into a JSON tool call.\n"
    "Request: look up the weather in Paris\n"
    'Call: {"name": "search", "arguments": {"query": "weather in Paris"}}\n'
    "Request: find the capital of Peru\n"
    "Call:"
)
N_TOKENS = 30

def check(text):
    try:
        parsed = json.loads(text)
    except Exception:
        return False, False
    ok = (isinstance(parsed, dict) and set(parsed) == {"name", "arguments"} and parsed["name"] == "search"
          and isinstance(parsed["arguments"], dict) and set(parsed["arguments"]) == {"query"}
          and isinstance(parsed["arguments"]["query"], str))
    return True, ok

try:
    import vllm
    from vllm import LLM, SamplingParams
    from vllm.sampling_params import StructuredOutputsParams
    result["vllm_version"] = vllm.__version__

    llm = LLM(model=model_id, **LLM_KWARGS)
    result["loaded"] = True

    free = SamplingParams(temperature=0.0, max_tokens=N_TOKENS, ignore_eos=True)
    result["free_text"] = llm.generate([PROMPT], free)[0].outputs[0].text   # also the warm-up
    times = []
    for _ in range(10):
        t0 = time.perf_counter()
        llm.generate([PROMPT], free, use_tqdm=False)
        times.append((time.perf_counter() - t0) * 1000)
    result["latency_batch1"] = {
        "n_new_tokens": N_TOKENS,
        "p50_ms_request": round(statistics.median(times), 1),
        "p50_ms_per_token": round(statistics.median(times) / N_TOKENS, 2),
        "max_ms_per_token": round(max(times) / N_TOKENS, 2),
        "runs_ms": [round(t, 1) for t in times],
    }

    variants = {
        "no_whitespace_max64": (dict(disable_any_whitespace=True), 64),
        "default_whitespace_max256": ({}, 256),
    }
    result["structured"] = {}
    for name, (opts, max_tokens) in variants.items():
        try:
            params = SamplingParams(temperature=0.0, max_tokens=max_tokens,
                                    structured_outputs=StructuredOutputsParams(json=SCHEMA, **opts))
            out = llm.generate([PROMPT], params)[0].outputs[0]
            valid, matches = check(out.text)
            result["structured"][name] = {"text": out.text, "finish_reason": out.finish_reason,
                                          "n_tokens": len(out.token_ids), "valid_json": valid,
                                          "matches_schema": matches}
        except Exception:
            result["structured"][name] = {"error": traceback.format_exc()[-2000:]}
except Exception:
    result["error_tail"] = traceback.format_exc()[-4000:]
    traceback.print_exc()

json.dump(result, open(out_path, "w"), indent=2)

In [ ]:
results = load_results()
entry = results["versions"][PRIMARY_VLLM]
CUSTOM_KWARGS = {"hf_overrides": {"architectures": ["AttnDropLlamaForCausalLM"],
                                  "dropped_attn_layers": DROPPED}}

for key, model_id, extra in [("control_extras", CONTROL_ID, None),
                             ("target_custom_class_extras", TARGET_ID, CUSTOM_KWARGS)]:
    if key not in entry:
        entry[key] = run_probe(model_id, "extras", PRIMARY_VLLM, tag_suffix=f"_{key}",
                               extra_kwargs=extra, script="vllm_probe_extra.py")
        save_results(results)
    r = entry[key]
    print(f"[{key}] loaded={r['loaded']} log={r['log']}")
    if not r["loaded"]:
        print(json.dumps(r.get("root_cause"), indent=2)[:3000])

In [ ]:
results = load_results()
entry = results["versions"][PRIMARY_VLLM]
ctrl, cust = entry.get("control_extras", {}), entry.get("target_custom_class_extras", {})

lines = [f"# Same-engine comparison (vLLM {PRIMARY_VLLM}, {entry['env'].get('gpu')}, float16, enforce_eager)", ""]
lines.append("| | Control (28 attention blocks) | attn-drop-6, custom class (22) |")
lines.append("|---|---|---|")
lat = lambda r, k: (r.get("latency_batch1") or {}).get(k)
lines.append(f"| p50 ms/token | {lat(ctrl, 'p50_ms_per_token')} | {lat(cust, 'p50_ms_per_token')} |")
lines.append(f"| p50 ms per 30-token request | {lat(ctrl, 'p50_ms_request')} | {lat(cust, 'p50_ms_request')} |")
lines.append(f"| max ms/token | {lat(ctrl, 'max_ms_per_token')} | {lat(cust, 'max_ms_per_token')} |")
if lat(ctrl, "p50_ms_per_token") and lat(cust, "p50_ms_per_token"):
    lines += ["", f"Latency ratio custom / control: {lat(cust, 'p50_ms_per_token') / lat(ctrl, 'p50_ms_per_token'):.3f}"]
lines.append("")
for name in ["no_whitespace_max64", "default_whitespace_max256"]:
    lines.append(f"## Structured outputs: {name}")
    for label, r in [("control", ctrl), ("custom class", cust)]:
        s = (r.get("structured") or {}).get(name)
        if not s:
            lines.append(f"- {label}: not run"); continue
        if "error" in s:
            lines.append(f"- {label}: ERROR {s['error'][-400:]!r}"); continue
        lines.append(f"- {label}: valid JSON={s['valid_json']}, matches schema={s['matches_schema']}, "
                     f"finish={s['finish_reason']}, tokens={s['n_tokens']}, text={s['text']!r}")
    lines.append("")

extras_report = "\n".join(lines)
(WORK_DIR / "report_extras.md").write_text(extras_report)
print(extras_report)

## Final report

Run on 2026-10-04, Colab **NVIDIA L4** (compute capability 8.9), float16 everywhere, one attempt per configuration.

**Environment**

| vLLM | torch | transformers | CUDA (torch / driver) | GPU |
|---|---|---|---|---|
| 0.30.0 | 2.13.0+cu130 | 5.17.0 | 13.0 / 13.0 | NVIDIA L4 |

Installing vLLM 0.30.0 changed torch, and the notebook flagged it. Every load ran in a fresh subprocess. No other vLLM version was needed because the control loaded on 0.30.0.

Shared settings: `dtype="half"`, `max_model_len=2048`, `gpu_memory_utilization=0.85`, `enforce_eager=True`, `trust_remote_code=True`.

### 1. Loading

| Configuration | Loads | KV memory | KV tokens | Bytes/token |
|---|---|---|---|---|
| Control, `unsloth/Llama-3.2-3B` | yes | 11.97 GiB | 112,016 | 114,740 (formula, 28 blocks: 114,688) |
| attn-drop-6, native vLLM Llama | **no** | n/a | n/a | n/a |
| attn-drop-6, Transformers backend (`model_impl="transformers"`) | **no** | n/a | n/a | n/a |
| attn-drop-6, custom vLLM class (plugin) | **yes** | 12.25 GiB | 145,920 | **90,141** (formula, 22 blocks: 90,112) |

**Why the first two fail.** Both stop with the same first error, after the shards were read (log line 150 native, 156 Transformers backend):

```
ValueError: Following weights were not initialized from checkpoint:
{'model.layers.{18,20,21,22,23,24}.input_layernorm.weight'}
```

The checkpoint has no `input_layernorm` and no `q/k/v/o_proj` for those six blocks; block 18 only ships `post_attention_layernorm` and the MLP. Both paths build 28 full Llama blocks and look for those weights. For the Transformers backend this suggests it instantiates the standard Llama class from `architectures` rather than the repo's `PrunedLlamaForCausalLM`. That is inferred from the identical error, not confirmed in the vLLM source.

**Custom vLLM class.** It is a vLLM plugin that reuses vLLM's `LlamaForCausalLM` with a decoder layer that, for the blocks in `dropped_attn_layers`, builds only `post_attention_layernorm` and the MLP and registers no `Attention` layer. The checkpoint is unchanged.

- **Correctness:** the greedy 30-token output is identical to the repo's Transformers class. That is one prompt only, not an accuracy evaluation.
- **KV cache:** 90,141 bytes/token, a ratio of 1.000 against the 22-block formula and 0.786 against 28. vLLM reserves KV cache only for the 22 attention blocks.
- **Tokens:** 145,920 cached tokens versus 112,016, which is 30% more. Available KV memory also rose by 0.28 GiB, which matches the attention weights removed: 6 blocks × about 25.2M parameters × 2 bytes.

### 2. Latency, same engine (batch 1, 30 tokens, median of 10 after a warm-up)

| | Control | attn-drop-6, custom class |
|---|---|---|
| p50 ms/token | 26.59 | 25.28 |
| p50 ms per request (includes prefill) | 797.7 | 758.3 |

The pruned model is **about 5% faster** (ratio 0.951). That matches the drop in parameters, 3.06B vs about 3.21B (ratio 0.953), which is what you would expect if batch-1 decode is limited by memory bandwidth. That reading is consistent with the data but not proven by it. Context here is short (around 60 tokens), so the KV saving barely affects speed; its effect at long context or high concurrency was not measured.

### 3. Structured outputs (`StructuredOutputsParams(json=SCHEMA)`, greedy)

| Variant | Control | attn-drop-6, custom class |
|---|---|---|
| `disable_any_whitespace=True`, `max_tokens=64` | valid, matches schema, 19 tokens, `stop` | **invalid**, 64 tokens, `length` |
| default whitespace, `max_tokens=256` | valid, matches schema, 19 tokens, `stop` | **invalid**, 256 tokens, `length` |

- **Control output:** `{"name": "search", "arguments": {"query": "capital of Peru"} }`.
- **Pruned model output:** `{"name": "search", "arguments": {"query": "capital of Peru"}` followed by a newline and spaces until the token limit.
- **The grammar is active on the pruned model.** It removes the leading space that the free-text output has, and it blocks the `Request:` line the model writes in free text. But after the inner `}` it lets the model emit whitespace, and the model never closes the outer object.
- **The custom class is not the likely cause.** Without a grammar, the pruned model also closes only one brace in Transformers, and the custom class matched that output token for token. This looks like a behaviour change caused by the pruning, observed on one prompt.
- **`disable_any_whitespace=True` had no visible effect on either model:** each model gave the same text in both variants. The active structured-output backend was not recorded, so why the flag did nothing is not established.

### What section 12.6 can honestly claim

On vLLM 0.30.0 (L4, float16), a Llama-3.2-3B with attention removed from 6 of 28 blocks does not load as is, neither through vLLM's native Llama nor through its Transformers backend. With a small custom model class registered as a vLLM plugin, it loads, matches the Transformers output on the test prompt, and gets KV cache only for its 22 attention blocks: 90,141 bytes/token as the formula predicts, 30% more tokens in the same memory. At batch 1 and short context it is about 5% faster than the unpruned model in the same engine. Under the same JSON grammar the unpruned model returns valid JSON, but the pruned one pads with whitespace and never closes the object. Serving support is necessary but not sufficient: the pruning changed the model's behaviour in a way the grammar did not catch.

**Untested:** other vLLM versions and the T4; which structured-output backend was used and why `disable_any_whitespace` had no effect; more prompts, both for structured outputs and for quality; throughput at long context and high concurrency, where the KV saving should matter most; CUDA graphs (`enforce_eager=False`); tensor parallelism.